# dbt Transform Pipeline

Runs dbt commands (run, test, compile) against a Fabric Lakehouse.
Parameters are injected by the Fabric Data Pipeline at runtime.


In [ ]:
# Pipeline parameters (injected at runtime by Fabric Data Pipeline)
command = "run"
models = ""
project_dir = "/tmp/dbt_project"
dbt_subdir = ""          # Optional sub-directory within cloned repo where dbt project lives (e.g. ".dq")
repo_url = ""
repo_branch = "main"
vault_url = "https://data-eng-key-vault.vault.azure.net/"
github_pat_secret = "github-pat"  # LEGACY: secret name for Key Vault lookup
github_token = ""  # NEW: actual token passed directly (preferred)
lakehouse_name = ""
lakehouse_id = ""
schema_name = "dbo"
workspace_id = ""



In [ ]:
# Install vd-dbt-fabricspark adapter
!pip install vd-dbt-fabricspark -q


In [ ]:
# Set environment variables required by dbt profiles.yml
import os

os.environ["LAKEHOUSE"] = lakehouse_name
os.environ["LAKEHOUSE_ID"] = lakehouse_id
os.environ["SCHEMA"] = schema_name
os.environ["WORKSPACE_ID"] = workspace_id
os.environ["DBT_JOB_NAME"] = f"dbt_{command}_{models}".replace(" ", "_") if models else f"dbt_{command}"

print(f"Environment: LAKEHOUSE={lakehouse_name}, SCHEMA={schema_name}, WORKSPACE_ID={workspace_id}")


In [ ]:
# Clone dbt project from Git
import subprocess, shutil, os

if not repo_url:
    raise ValueError("repo_url parameter is required")

# Clean previous clone
shutil.rmtree(project_dir, ignore_errors=True)
os.makedirs(os.path.dirname(project_dir), exist_ok=True)

# Authenticate via token (direct or Key Vault)
if github_token:
    print("Using GitHub token from pipeline parameter")
    pat = github_token
else:
    print("Using GitHub PAT from Key Vault (legacy mode)")
    pat = notebookutils.credentials.getSecret(vault_url, github_pat_secret)

auth_url = repo_url.replace("https://", f"https://{pat}@")

clone_args = ["git", "clone", "--depth", "1"]
if repo_branch:
    clone_args.extend(["-b", repo_branch])
clone_args.extend([auth_url, project_dir])

result = subprocess.run(clone_args, capture_output=True, text=True, timeout=120, check=True)
print(f"Cloned {repo_url} (branch: {repo_branch}) into {project_dir}")



In [ ]:
# Execute dbt command and persist JSON logs + artifacts to lakehouse
import subprocess, time, shutil, os, yaml, re, json
from datetime import datetime, timezone

start_time = time.time()

# Set up lakehouse log directory for --log-path and artifact persistence
run_ts_path = datetime.now(timezone.utc).strftime('%Y/%m/%d')
job_name = os.environ.get('DBT_JOB_NAME', 'unknown')
log_dir = f'/lakehouse/default/Files/logs/dbt/{run_ts_path}'
os.makedirs(log_dir, exist_ok=True)

# When dbt_subdir is set the dbt project lives inside a subdirectory of the
# cloned repo (e.g. ".dq" for DQ tests).  The clone itself always goes to
# project_dir; dbt is pointed at project_dir/dbt_subdir.
dbt_project_dir = os.path.join(project_dir, dbt_subdir) if dbt_subdir else project_dir

# Install dbt packages (elementary, dbt_utils, etc.)
deps_proc = subprocess.run(
    ["dbt", "deps", "--project-dir", dbt_project_dir, "--profiles-dir", dbt_project_dir],
    capture_output=True, text=True, cwd=dbt_project_dir
)
print(deps_proc.stdout)
if deps_proc.returncode != 0:
    print(f"Warning: dbt deps failed: {deps_proc.stderr}")

# ── Bootstrap elementary tables if not present ──
# Reads the elementary schema name from dbt_project.yml, then uses
# `dbt show --inline` to check if tables exist in that schema.
# When the schema is empty (first run), `dbt run -s elementary`
# materialises the required monitoring tables.
try:
    dbt_project_path = os.path.join(dbt_project_dir, "dbt_project.yml")
    elementary_schema = None
    if os.path.exists(dbt_project_path):
        with open(dbt_project_path) as f:
            dbt_cfg = yaml.safe_load(f)
        # Elementary schema is at models.elementary.+schema in dbt_project.yml
        elementary_schema = (
            dbt_cfg.get("models", {})
            .get("elementary", {})
            .get("+schema")
        )
    if elementary_schema:
        # Validate schema name to prevent injection
        if not re.match(r'^[a-zA-Z0-9_]+$', elementary_schema):
            raise ValueError(f"Invalid elementary schema name: {elementary_schema}")
        check_proc = subprocess.run(
            [
                "dbt", "show", "--inline",
                f"select count(*) as cnt from INFORMATION_SCHEMA.TABLES where TABLE_SCHEMA = '{elementary_schema}'",
                "--project-dir", dbt_project_dir, "--profiles-dir", dbt_project_dir,
            ],
            capture_output=True, text=True, cwd=dbt_project_dir
        )
        # dbt show outputs a table — look for a row with "| 0 |" meaning zero tables
        has_tables = check_proc.returncode == 0 and "| 0" not in check_proc.stdout
        if not has_tables:
            print(f"Elementary schema '{elementary_schema}' is empty — bootstrapping with dbt run -s elementary")
            elem_proc = subprocess.run(
                ["dbt", "run", "-s", "elementary", "--project-dir", dbt_project_dir, "--profiles-dir", dbt_project_dir],
                capture_output=True, text=True, cwd=dbt_project_dir
            )
            print(elem_proc.stdout)
            if elem_proc.returncode != 0:
                print(f"Warning: elementary bootstrap failed: {elem_proc.stderr}")
            else:
                print("Elementary tables bootstrapped successfully")
        else:
            print(f"Elementary schema '{elementary_schema}' already has tables — skipping bootstrap")
    else:
        print("No elementary schema found in dbt_project.yml — skipping bootstrap")
except Exception as e:
    print(f"Warning: elementary bootstrap check failed: {e}")

# Run dbt with JSON-formatted logs written directly to lakehouse via --log-path
dbt_args = [
    "dbt", command,
    "--project-dir", dbt_project_dir,
    "--profiles-dir", dbt_project_dir,
    "--log-format", "json",
    "--log-path", log_dir,
]
if models:
    dbt_args.extend(["--select", models])

proc = subprocess.run(dbt_args, capture_output=True, text=True, cwd=dbt_project_dir)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)

duration = int((time.time() - start_time) * 1000)
print(f"Duration: {duration}ms")
print(f"ExitCode: {proc.returncode}")

try:
    if proc.returncode != 0:
        full_output = proc.stderr or proc.stdout or "No output captured"
        
        # Extract lines after first occurrence of "error" (case-insensitive)
        error_section = full_output
        lines = full_output.split('\n')
        for i, line in enumerate(lines):
            if 'error' in line.lower():
                error_section = '\n'.join(lines[i:])
                break
        
        raise RuntimeError(f"dbt {command} failed with exit code {proc.returncode}\n\n{error_section}")
finally:
    # ── Persist logs and artifacts into an invocation_id folder ──
    # dbt generates a unique invocation_id per run. We read it from
    # target/run_results.json and use it as the folder name so the
    # alert agent can reliably locate artifacts for a specific run.
    #
    # Structure: logs/dbt/{YYYY}/{MM}/{DD}/{invocation_id}/
    #   ├── dbt.log
    #   ├── run_results.json
    #   └── manifest.json

    invocation_id = None
    results_src = os.path.join(dbt_project_dir, 'target', 'run_results.json')

    # Extract invocation_id from run_results.json
    try:
        if os.path.exists(results_src):
            with open(results_src) as f:
                run_results = json.load(f)
            invocation_id = run_results.get('metadata', {}).get('invocation_id')
    except Exception as e:
        print(f"Warning: could not read invocation_id from run_results.json: {e}")

    # Fallback folder name when invocation_id is unavailable (crash before results written)
    run_id = datetime.now(timezone.utc).strftime('%H%M%S')
    folder_name = invocation_id or f"{job_name}_{run_id}"
    inv_dir = f'{log_dir}/{folder_name}'
    os.makedirs(inv_dir, exist_ok=True)

    # Move dbt.log (written by --log-path) into invocation folder
    try:
        dbt_log_src = os.path.join(log_dir, 'dbt.log')
        log_dest = f'{inv_dir}/dbt.log'
        if os.path.exists(dbt_log_src):
            os.rename(dbt_log_src, log_dest)
            print(f"dbt JSON logs persisted to lakehouse: {log_dest}")
        else:
            # Fallback: write captured stdout/stderr when --log-path did not produce a file
            with open(log_dest, 'w') as f:
                f.write(f"# dbt {command} | duration={duration}ms | exit_code={proc.returncode}\n")
                f.write(f"# timestamp={datetime.now(timezone.utc).isoformat()}\n\n")
                f.write(proc.stdout or '')
                if proc.stderr:
                    f.write('\n--- STDERR ---\n')
                    f.write(proc.stderr)
            print(f"Captured output saved to lakehouse: {log_dest}")
    except Exception as log_err:
        print(f"Warning: failed to persist logs to lakehouse: {log_err}")

    # Copy run_results.json and manifest.json into invocation folder
    try:
        if os.path.exists(results_src):
            shutil.copy2(results_src, f'{inv_dir}/run_results.json')
            print(f"run_results.json persisted to lakehouse: {inv_dir}/run_results.json")

        manifest_src = os.path.join(dbt_project_dir, 'target', 'manifest.json')
        if os.path.exists(manifest_src):
            shutil.copy2(manifest_src, f'{inv_dir}/manifest.json')
            print(f"manifest.json persisted to lakehouse: {inv_dir}/manifest.json")
    except Exception as artifact_err:
        print(f"Warning: failed to persist dbt artifacts: {artifact_err}")

    print(f"Invocation folder: {inv_dir}")

